<a href="https://colab.research.google.com/github/hvsya/AI-Project_PlantVillage/blob/main/DataModeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Modelling — Plant Disease Classification
**Role:** Data Scientist  
**Models:** ResNet50 · DenseNet121 · MobileNetV3Small  
**Epochs:** 50 each  
**Metrics:** Accuracy · mAP (mean Average Precision)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/AI_Project')

Mounted at /content/drive


## 1. Environment Setup

In [2]:
import os
import time
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV3Small
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Dataset Loading
Continues from the `final_plantvillage_dataset/` folder created in DataPreparation.ipynb.

In [3]:
OUTPUT_DIR   = "final_plantvillage_dataset"
TARGET_SIZE  = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 50
NUM_CLASSES  = 10   # Apple×4, Blueberry×1, Corn×1, Grape×1, Peach×1, Tomato×2

# ── Load splits ──────────────────────────────────────────────────────────────
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(OUTPUT_DIR, 'train'),
    image_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=42
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(OUTPUT_DIR, 'val'),
    image_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(OUTPUT_DIR, 'test'),
    image_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

CLASS_NAMES = train_ds.class_names
print("Classes:", CLASS_NAMES)
print("Train batches:", len(train_ds))
print("Val   batches:", len(val_ds))
print("Test  batches:", len(test_ds))

Found 4200 files belonging to 10 classes.
Found 900 files belonging to 10 classes.
Found 900 files belonging to 10 classes.
Classes: ['Apple_Black_rot', 'Apple_Cedar_apple_rust', 'Apple_healthy', 'Apple_scab', 'Blueberry_healthy', 'Corn_Common_rust', 'Grape_Black_rot', 'Peach_Bacterial_spot', 'Tomato_Late_blight', 'Tomato_healthy']
Train batches: 132
Val   batches: 29
Test  batches: 29


## 3. Data Augmentation & Preprocessing Pipeline

In [4]:
# Augmentation applied only during training
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.10),
    layers.RandomBrightness(0.10),
    layers.RandomContrast(0.10),
], name="augmentation")

def preprocess(images, labels):
    """Cast to float32; backbone-specific rescaling is applied inside each model."""
    return tf.cast(images, tf.float32), labels

AUTOTUNE = tf.data.AUTOTUNE

train_ds_prep = (
    train_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

val_ds_prep = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds_prep = test_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print("Pipelines ready.")

Pipelines ready.


## 4. Helper — Build Transfer-Learning Model

In [5]:
def build_model(backbone_fn, preprocess_input, model_name, num_classes=NUM_CLASSES):
    """
    Wraps a Keras application backbone with:
      - backbone-specific preprocess_input
      - frozen base weights (ImageNet) — transfer learning feature extraction
      - GlobalAveragePooling2D head
      - Dropout (0.4) + Dense softmax
    Returns compiled Keras model.
    """
    inputs = keras.Input(shape=(224, 224, 3), name="input_image")

    # Backbone-specific preprocessing (normalisation)
    x = layers.Lambda(preprocess_input, name="preprocess")(inputs)

    # Base CNN — ImageNet weights, top excluded, frozen for transfer learning
    base = backbone_fn(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    base.trainable = False   # Frozen — feature extraction only

    # Custom head
    x = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x = layers.BatchNormalization(name="bn_head")(x)
    x = layers.Dense(256, activation="relu", name="fc1")(x)
    x = layers.Dropout(0.4, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = Model(inputs=inputs, outputs=outputs, name=model_name)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def compute_map(model, dataset, class_names):
    """Compute mean Average Precision (one-vs-rest) over the given dataset."""
    all_labels, all_probs = [], []
    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    all_labels = np.concatenate(all_labels, axis=0)
    all_probs  = np.concatenate(all_probs,  axis=0)
    aps = []
    for i in range(len(class_names)):
        ap = average_precision_score(all_labels[:, i], all_probs[:, i])
        aps.append(ap)
    return np.mean(aps), aps, all_labels, all_probs


# Storage for comparison
results = {}

---
## 5. Model 1 — ResNet50
Deep residual network with skip connections, ~25M parameters.

In [6]:
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

resnet_model = build_model(
    backbone_fn=ResNet50,
    preprocess_input=resnet_preprocess,
    model_name="ResNet50_PlantDisease"
)
resnet_model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "ResNet50_PlantDisease"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ preprocess (Lambda) │ (None, 224, 224,  │          0 │ input_image[0][0] │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ preprocess[0][0]  │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c

 Total params: 24,123,018 (92.02 MB)

 Trainable params: 531,210 (2.03 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

In [7]:
callbacks_resnet = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint("best_resnet50.keras", save_best_only=True, monitor='val_accuracy', verbose=0)
]

print("=== ResNet50 — Training (50 epochs) ===")
start = time.time()
hist_rn = resnet_model.fit(
    train_ds_prep, validation_data=val_ds_prep,
    epochs=EPOCHS, callbacks=callbacks_resnet
)
resnet_time = time.time() - start
print(f"\nResNet50 total training time: {resnet_time/60:.2f} min")

resnet_history = hist_rn.history

=== ResNet50 — Training (50 epochs) ===
Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1631s 12s/step - accuracy: 0.3121 - loss: 2.5381 - val_accuracy: 0.4244 - val_loss: 1.7383 - learning_rate: 0.0010
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 65s 487ms/step - accuracy: 0.4098 - loss: 1.9005 - val_accuracy: 0.4744 - val_loss: 1.6842 - learning_rate: 0.0010
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 63s 467ms/step - accuracy: 0.4569 - loss: 1.5978 - val_accuracy: 0.4422 - val_loss: 1.6506 - learning_rate: 0.0010
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 82s 470ms/step - accuracy: 0.4895 - loss: 1.4526 - val_accuracy: 0.4522 - val_loss: 1.6573 - learning_rate: 0.0010
Epoch 5/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 63s 469ms/step - accuracy: 0.5257 - loss: 1.3211 - val_accuracy: 0.4456 - val_loss: 1.6170 - learning_rate: 0.0010
Epoch 6/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 80s 458ms/step - accuracy: 0.5405 - loss: 1.2656 - val_accuracy: 0.4556 - val_loss: 1.6119 - learning_rate: 0.0010
Epoch 7/50
132/132 ━━━━━━━━━━━

In [ ]:
# Evaluate ResNet50
resnet_loss, resnet_acc = resnet_model.evaluate(test_ds_prep, verbose=0)
resnet_map, resnet_aps, rn_labels, rn_probs = compute_map(resnet_model, test_ds_prep, CLASS_NAMES)

print(f"ResNet50  |  Test Accuracy: {resnet_acc:.4f}  |  mAP: {resnet_map:.4f}  |  Time: {resnet_time/60:.2f} min")

results['ResNet50'] = {
    'history': resnet_history,
    'test_acc': resnet_acc,
    'test_loss': resnet_loss,
    'mAP': resnet_map,
    'per_class_AP': resnet_aps,
    'train_time_min': resnet_time / 60,
    'labels': rn_labels,
    'probs': rn_probs,
    'params': resnet_model.count_params()
}

---
## 6. Model 2 — DenseNet121
Dense connectivity pattern where each layer receives feature maps from all preceding layers, ~8M parameters.

In [ ]:
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

densenet_model = build_model(
    backbone_fn=DenseNet121,
    preprocess_input=densenet_preprocess,
    model_name="DenseNet121_PlantDisease"
)
densenet_model.summary()

In [ ]:
callbacks_densenet = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint("best_densenet121.keras", save_best_only=True, monitor='val_accuracy', verbose=0)
]

print("=== DenseNet121 — Training (50 epochs) ===")
start = time.time()
hist_dn = densenet_model.fit(
    train_ds_prep, validation_data=val_ds_prep,
    epochs=EPOCHS, callbacks=callbacks_densenet
)
densenet_time = time.time() - start
print(f"\nDenseNet121 total training time: {densenet_time/60:.2f} min")

densenet_history = hist_dn.history

In [ ]:
densenet_loss, densenet_acc = densenet_model.evaluate(test_ds_prep, verbose=0)
densenet_map, densenet_aps, dn_labels, dn_probs = compute_map(densenet_model, test_ds_prep, CLASS_NAMES)

print(f"DenseNet121 |  Test Accuracy: {densenet_acc:.4f}  |  mAP: {densenet_map:.4f}  |  Time: {densenet_time/60:.2f} min")

results['DenseNet121'] = {
    'history': densenet_history,
    'test_acc': densenet_acc,
    'test_loss': densenet_loss,
    'mAP': densenet_map,
    'per_class_AP': densenet_aps,
    'train_time_min': densenet_time / 60,
    'labels': dn_labels,
    'probs': dn_probs,
    'params': densenet_model.count_params()
}

---
## 7. Model 3 — MobileNetV3Small
Lightweight architecture designed for edge/mobile inference, ~2.5M parameters.

In [ ]:
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenet_preprocess

mobilenet_model = build_model(
    backbone_fn=MobileNetV3Small,
    preprocess_input=mobilenet_preprocess,
    model_name="MobileNetV3Small_PlantDisease"
)
mobilenet_model.summary()

In [ ]:
callbacks_mobilenet = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint("best_mobilenetv3.keras", save_best_only=True, monitor='val_accuracy', verbose=0)
]

print("=== MobileNetV3Small — Training (50 epochs) ===")
start = time.time()
hist_mn = mobilenet_model.fit(
    train_ds_prep, validation_data=val_ds_prep,
    epochs=EPOCHS, callbacks=callbacks_mobilenet
)
mobilenet_time = time.time() - start
print(f"\nMobileNetV3Small total training time: {mobilenet_time/60:.2f} min")

mobilenet_history = hist_mn.history

In [ ]:
mobilenet_loss, mobilenet_acc = mobilenet_model.evaluate(test_ds_prep, verbose=0)
mobilenet_map, mobilenet_aps, mn_labels, mn_probs = compute_map(mobilenet_model, test_ds_prep, CLASS_NAMES)

print(f"MobileNetV3 |  Test Accuracy: {mobilenet_acc:.4f}  |  mAP: {mobilenet_map:.4f}  |  Time: {mobilenet_time/60:.2f} min")

results['MobileNetV3Small'] = {
    'history': mobilenet_history,
    'test_acc': mobilenet_acc,
    'test_loss': mobilenet_loss,
    'mAP': mobilenet_map,
    'per_class_AP': mobilenet_aps,
    'train_time_min': mobilenet_time / 60,
    'labels': mn_labels,
    'probs': mn_probs,
    'params': mobilenet_model.count_params()
}

---
## 8. Summary Table

In [ ]:
import pandas as pd

summary_data = []
for name, r in results.items():
    summary_data.append({
        'Model': name,
        'Parameters': f"{r['params']:,}",
        'Test Accuracy': f"{r['test_acc']:.4f}",
        'mAP': f"{r['mAP']:.4f}",
        'Training Time (min)': f"{r['train_time_min']:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Save results (without numpy arrays) for use in visualization notebook
save_results = {}
for name, r in results.items():
    save_results[name] = {
        'history': r['history'],
        'test_acc': float(r['test_acc']),
        'test_loss': float(r['test_loss']),
        'mAP': float(r['mAP']),
        'per_class_AP': [float(x) for x in r['per_class_AP']],
        'train_time_min': float(r['train_time_min']),
        'params': int(r['params'])
    }

with open('model_results.json', 'w') as f:
    json.dump({'class_names': CLASS_NAMES, 'results': save_results}, f, indent=2)
print("\nResults saved to model_results.json")